# Clase 13 · Fraude: detección de anomalías y grafos

**Machine Learning** · Programa de Especialización Data Scientist · Sesión 14 de 18

Docente: **Josef Renato Rodriguez**

---

## La pregunta de hoy

Usamos el mismo transaccional de la Clase 12 (439 mil compras con tarjeta de los clientes de las Clases 9 y 10). Hoy el
que pregunta es el **equipo de prevención de fraude**:

> **¿Qué compras debemos revisar hoy, y quién está detrás de ellas?**

Son dos preguntas distintas y cada una tiene su herramienta:

| Pregunta | Herramienta | Mira |
|---|---|---|
| ¿Qué compra es **rara**? | **Isolation Forest** (y LOF) | cada transacción contra el comportamiento normal |
| ¿Quién está **conectado** con quién? | **Grafos** (NetworkX) | dispositivos compartidos y comercios en común |

## Qué vas a hacer hoy

| Bloque | Contenido | Min |
|---|---|---|
| A | Por qué el fraude no es una clasificación más | 15 |
| B | Variables: el monto no dice nada, **el monto comparado con el cliente** sí | 25 |
| C | **Isolation Forest**, LOF y una regla de negocio, evaluados con **precisión@k** | 30 |
| D | **Grafos**: dispositivos compartidos y **punto de compromiso** | 35 |
| E | Variables de grafo dentro del modelo de anomalías | 10 |

---
## Celda 0 · Librerías y datos

La celda lee cuatro archivos del repositorio del curso ([Super_ML_course](https://github.com/josefrodrim/Super_ML_course)):

- `transacciones_tarjetas.csv.gz` y `comercios.csv` (carpeta `Clase_12/Data`): el transaccional de la clase pasada.
- `fraude_etiquetas.csv` (carpeta `Clase_12/Data`): las compras que el banco **confirmó** como fraude. Solo la usamos
  para **evaluar**: ningún modelo de anomalías la ve al entrenar.
- `reclamos_fraude.csv` (carpeta `Clase_13/Data`): los reclamos de clientes por cargos que no reconocen, **con su fecha**.
  Es lo que el banco sabe en cada momento.

Si abres el notebook dentro del repositorio, los lee de tu disco; si no (Colab), los baja de GitHub.
Si `networkx` no está instalado, la celda lo instala (en Colab ya viene).

In [ ]:
import sys, subprocess, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
try:
    import networkx as nx
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "networkx"], check=True)
    import networkx as nx
import warnings
warnings.filterwarnings("ignore")

SEED = 42
from matplotlib.ticker import PercentFormatter, FuncFormatter
from matplotlib.lines import Line2D

NAVY, BLUE, MAG, GREEN, GRAY, ORANGE = "#0A2559", "#1A56E8", "#E6115E", "#12B886", "#5B6B87", "#E08A00"
TINTA, TINTA_2, LINEA = "#1F2A44", "#5B6B87", "#D5DAE3"
# Cada patrón de fraude tiene SIEMPRE el mismo color en todos los gráficos
COLOR_PATRON = {"tarjeta_clonada": BLUE, "comercio_comprometido": MAG, "prueba_tarjeta": ORANGE}
NOMBRE_PATRON = {"tarjeta_clonada": "Tarjeta clonada", "comercio_comprometido": "Comercio comprometido",
                 "prueba_tarjeta": "Prueba de tarjeta"}
plt.rcParams.update({
    "figure.figsize": (8, 4.5), "figure.dpi": 110, "savefig.dpi": 150,
    "font.size": 10.5, "text.color": TINTA, "axes.labelcolor": TINTA_2,
    "xtick.color": TINTA_2, "ytick.color": TINTA_2,
    "axes.edgecolor": LINEA, "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": LINEA, "grid.alpha": 0.6, "grid.linewidth": 0.6,
    "axes.axisbelow": True,
    "axes.titlesize": 12.5, "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlepad": 22,
    "legend.frameon": False, "legend.fontsize": 9.5,
})
MILES = FuncFormatter(lambda v, _: f"{v:,.0f}".replace(",", " "))

def subtitulo(ax, texto):
    """Línea gris bajo el título: qué mirar en el gráfico."""
    ax.text(0, 1.015, texto, transform=ax.transAxes, fontsize=9.5, color=TINTA_2, va="bottom")

def check(nombre, obtenido, esperado, tol=1e-3):
    if obtenido is None:
        print(f"[ ] {nombre}: todavía no calculaste nada")
        return False
    ok = abs(float(obtenido) - float(esperado)) <= tol
    print(f"{'[OK]' if ok else '[X ]'} {nombre}")
    print(f"     tu resultado: {float(obtenido):.4f}   |   esperado: {float(esperado):.4f}")
    return ok

def check_bool(nombre, cond, pista=""):
    print(f"{'[OK]' if cond else '[X ]'} {nombre}")
    if not cond and pista:
        print(f"     pista: {pista}")
    return bool(cond)

from pathlib import Path
URL_REPO = "https://raw.githubusercontent.com/josefrodrim/Super_ML_course/main/"

def leer(ruta, **kw):
    """Lee el archivo del repositorio local si existe; si no (Colab), lo baja de GitHub."""
    for base in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        if (base / ruta).exists():
            return pd.read_csv(base / ruta, **kw)
    return pd.read_csv(URL_REPO + ruta, **kw)

tx = leer("Clase_12/Data/transacciones_tarjetas.csv.gz", compression="gzip", parse_dates=["fecha_hora"])
comercios = leer("Clase_12/Data/comercios.csv")
etiquetas = leer("Clase_12/Data/fraude_etiquetas.csv")
reclamos = leer("Clase_13/Data/reclamos_fraude.csv", parse_dates=["fecha_reclamo", "fecha_primer_cargo"])

tx = (tx.merge(comercios, on="id_comercio", how="left")
        .sort_values(["fecha_hora", "id_transaccion"]).reset_index(drop=True))
tx["es_fraude"] = tx["id_transaccion"].isin(etiquetas["id_transaccion"]).astype(int)

INICIO_2026 = pd.Timestamp("2026-01-01")
print(f"{len(tx):,} compras · {tx['id_cliente'].nunique():,} clientes · {tx['es_fraude'].sum()} fraudes confirmados".replace(",", " "))
tx.head()

---
# Bloque A · El fraude no es una clasificación más  ·  15 min

En las Clases 6 a 9 teníamos un target con miles de casos de cada clase. El fraude tiene tres problemas que cambian el
enfoque:

1. **Es rarísimo.** Hay pocas etiquetas y un modelo supervisado aprende muy poco de ellas.
2. **Las etiquetas llegan tarde.** El banco se entera cuando el cliente reclama, días o semanas después.
3. **El patrón cambia.** El defraudador se adapta: lo que funcionó el año pasado no es lo que va a pasar este mes.

Por eso se usa **detección de anomalías**: aprender cómo se ve lo **normal** y alertar lo que se aleja. No necesita
etiquetas para entrenar; las etiquetas se usan solo para **evaluar**.

### Demostración A.0 — Cuánto fraude hay

In [ ]:
# ── DEMOSTRACIÓN A.0 ──────────────────────────────────────────────────
print(f"Tasa de fraude: {tx['es_fraude'].mean():.4%}  →  1 de cada {1 / tx['es_fraude'].mean():,.0f} compras".replace(",", " "))
f = tx[tx["es_fraude"] == 1].merge(etiquetas[["id_transaccion", "patron"]], on="id_transaccion")
print(f.groupby("patron").agg(compras=("monto", "size"), clientes=("id_cliente", "nunique"),
                              monto_mediano=("monto", "median"), desde=("fecha_hora", "min"), hasta=("fecha_hora", "max")).round(0).to_string())

# todos los meses de la base, también los que no tienen fraude
meses = pd.period_range(tx["fecha_hora"].min(), tx["fecha_hora"].max(), freq="M")
por_mes = (f.groupby([f["fecha_hora"].dt.to_period("M"), "patron"]).size().unstack(fill_value=0)
             .reindex(index=meses, columns=list(COLOR_PATRON), fill_value=0))
MES = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]
etq = [f"{MES[p.month - 1]}\n{p.year}" if p.month in (1, 7) or i == 0 else MES[p.month - 1] for i, p in enumerate(meses)]

fig, ax = plt.subplots(figsize=(10, 4.2))
x = np.arange(len(meses)); base = np.zeros(len(meses))
for patron, color in COLOR_PATRON.items():
    ax.bar(x, por_mes[patron], bottom=base, color=color, width=0.7, edgecolor="white", linewidth=1.5,
           label=NOMBRE_PATRON[patron])
    base += por_mes[patron].values
for xi, tot in zip(x, base):
    if tot:
        ax.text(xi, tot + 2, f"{tot:.0f}", ha="center", va="bottom", fontsize=9, color=TINTA)
n_perfil = int((meses < pd.Period("2026-01", "M")).sum())
ax.axvspan(-0.5, n_perfil - 0.5, color=LINEA, alpha=0.35, lw=0)
ax.text((n_perfil - 1) / 2, base.max() * 0.55, "jul–dic 2025\nperfil del cliente\n(sin fraude confirmado)",
        ha="center", fontsize=9, color=TINTA_2)
ax.set_xticks(x, etq); ax.tick_params(axis="x", length=0)
ax.set_ylim(0, base.max() * 1.25); ax.set_ylabel("compras fraudulentas")
ax.set_title("Fraudes confirmados por mes")
subtitulo(ax, "Tres patrones que aparecen en momentos distintos de 2026")
ax.legend(loc="upper right", ncol=3)
plt.tight_layout(); plt.show()

Hay **tres patrones** de fraude en la base, y conviene conocerlos porque cada uno se detecta distinto:

| Patrón | Qué pasó |
|---|---|
| **Tarjeta clonada** | copiaron la banda o el chip y la usan presencialmente en otro país, muchas veces de madrugada |
| **Comercio comprometido** | un comercio de Lima filtró los datos de las tarjetas que pasaron por ahí; semanas después aparecen compras online |
| **Prueba de tarjeta** | el defraudador prueba la tarjeta con un monto chico (S/ 1 a 5) y, si pasa, hace compras grandes |

### Ejercicio A.1 — Tasa de fraude por canal y país

Crea la columna `tx["extranjero"]` (1 si `pais` no es `"Perú"`) y la tabla `tasas`: agrupando por `canal` y `extranjero`,
las columnas `compras` (cuántas), `fraudes` (suma de `es_fraude`) y `tasa` (promedio de `es_fraude`).

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
# tx["extranjero"] = ...
tasas = None

In [ ]:
# ── VERIFICACIÓN A.1 ──────────────────────────────────────────────────
def tz(canal, ext):
    try:
        return tasas.loc[(canal, ext), "tasa"]
    except Exception:
        return None
r = [check("tasa de fraude global", tx["es_fraude"].mean(), 0.001175, tol=1e-5),
     check("tasa: online en el extranjero", None if tasas is None else tz("online", 1), 0.8473, tol=1e-3),
     check("tasa: presencial en el extranjero", None if tasas is None else tz("presencial", 1), 0.0575, tol=1e-3)]
print()
print("A.1 OK" if all(r) else "Revisa A.1")

**Todo el fraude de esta base ocurre en comercios del extranjero**, pero no todas las compras en el extranjero son fraude:

- **Online en el extranjero:** 85 % es fraude. Una regla simple ya lo agarra.
- **Presencial en el extranjero:** solo 5.7 %. La mayoría son **clientes de viaje**. Si alertamos todas, el equipo revisa
  3 184 compras para encontrar 183 fraudes.

Ahí está el trabajo de hoy: distinguir al **viajero** del **clon**, y al comprador online de siempre del que usa una
tarjeta robada.

---
# Bloque B · Variables: comparar contra el propio cliente  ·  25 min

Un monto de S/ 2 000 es normal para un cliente Premium viajero y rarísimo para un Usuario de efectivo. El fraude casi nunca
se ve en el número **absoluto**; se ve en el número **comparado con la historia del cliente**.

### Ejercicio B.1 — Variables de contexto

| Variable | Definición |
|---|---|
| `log_monto` | `np.log1p(monto)` |
| `madrugada` | 1 si la hora de `fecha_hora` está entre 0 y 5 (incluidas) |
| `online` | 1 si `canal` es `"online"` |

(`extranjero` ya la creaste en A.1.)

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
# tx["log_monto"] = ...
# tx["madrugada"] = ...
# tx["online"] = ...

In [ ]:
# ── VERIFICACIÓN B.1 ──────────────────────────────────────────────────
def media(col):
    return tx[col].mean() if col in tx else None
r = [check("promedio de log_monto", media("log_monto"), 5.5868, tol=1e-3),
     check("% de compras de madrugada", media("madrugada"), 0.01505, tol=1e-4),
     check("% de compras online", media("online"), 0.3467, tol=1e-3)]
print()
print("B.1 OK" if all(r) else "Revisa B.1")

### Ejercicio B.2 — Variables relativas al cliente

| Variable | Definición |
|---|---|
| `ratio_monto` | `monto` dividido entre la **mediana** de las compras del cliente en el **perfil** (julio a diciembre de 2025). Si el cliente no tiene compras en el perfil, usa la mediana de todas las compras del perfil. |
| `comercio_nuevo` | 1 si es la **primera vez** que el cliente compra en ese comercio |
| `dispositivo_nuevo` | 1 si es una compra online desde un dispositivo que el cliente **nunca había usado** |

Pistas:
- El perfil son las compras con `fecha_hora < INICIO_2026`. Usar solo el pasado para describir al cliente es la misma
  regla de la fecha de corte de la Clase 12.
- «Primera vez» es `groupby([...]).cumcount() == 0`. Funciona porque `tx` está ordenado por fecha.
- En `dispositivo_nuevo`, las compras presenciales no tienen dispositivo: deben quedar en 0 (`& tx["id_dispositivo"].notna()`).

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
perfil = None    # compras de julio a diciembre de 2025
# tx["ratio_monto"] = ...
# tx["comercio_nuevo"] = ...
# tx["dispositivo_nuevo"] = ...

In [ ]:
# ── VERIFICACIÓN B.2 ──────────────────────────────────────────────────
r = [check("promedio de ratio_monto", media("ratio_monto"), 1.3255, tol=2e-3),
     check("mediana de ratio_monto en las compras fraudulentas", tx.loc[tx["es_fraude"] == 1, "ratio_monto"].median() if "ratio_monto" in tx else None, 8.4035, tol=0.01),
     check("% de compras en un comercio nuevo", media("comercio_nuevo"), 0.2592, tol=1e-3),
     check("% de compras desde un dispositivo nuevo", media("dispositivo_nuevo"), 0.01757, tol=2e-4)]
print()
print("B.2 OK" if all(r) else "Revisa B.2")

In [ ]:
# ── DEMOSTRACIÓN B.2b · El monto absoluto contra el monto relativo ──────
if "ratio_monto" in tx:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, col, bins, titulo, xlab in [
        (axes[0], "monto", np.logspace(0, 4.7, 50), "Monto absoluto", "monto (S/, escala log)"),
        (axes[1], "ratio_monto", np.logspace(-2, 2.3, 50), "Monto ÷ mediana del cliente", "ratio_monto (escala log)")]:
        for v, color, nombre in [(0, GRAY, "legítimas"), (1, MAG, "fraude")]:
            vals = tx.loc[tx["es_fraude"] == v, col].clip(bins[0], bins[-1])
            peso = np.full(len(vals), 1 / len(vals))          # % de las compras de cada grupo
            ax.hist(vals, bins=bins, weights=peso, histtype="stepfilled", alpha=0.25, color=color, lw=0)
            ax.hist(vals, bins=bins, weights=peso, histtype="step", color=color, lw=1.8, label=nombre)
        ax.set_xscale("log"); ax.set_xlabel(xlab); ax.set_title(titulo)
        ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0)); ax.set_ylabel("% de las compras del grupo")
    axes[1].axvline(1, color=TINTA_2, lw=0.8, ls="--")
    axes[1].text(1.15, axes[1].get_ylim()[1] * 0.92, "lo habitual\ndel cliente", fontsize=8.5, color=TINTA_2, va="top")
    axes[0].legend(loc="upper left")
    subtitulo(axes[0], "Fraude y compras legítimas se enciman")
    subtitulo(axes[1], "Comparado con el propio cliente, el fraude se separa")
    plt.tight_layout(); plt.show()

La compra fraudulenta típica es **8.4 veces** lo que gasta normalmente ese cliente. En monto absoluto, en cambio, se
confunde con las compras grandes de los clientes que gastan mucho.

### Demostración B.3 — Velocidad: compras en las últimas 24 horas

La prueba de tarjeta y el fraude por comercio comprometido llegan **en ráfaga**: varias compras en pocas horas. La variable
`tx_24h` cuenta cuántas compras hizo el cliente en las 24 horas **anteriores** a cada compra. Se calcula con una ventana
móvil de tiempo por cliente (`rolling("24h")`).

In [ ]:
# ── DEMOSTRACIÓN B.3 ──────────────────────────────────────────────────
tx = tx.sort_values(["id_cliente", "fecha_hora", "id_transaccion"])
tx["tx_24h"] = tx.groupby("id_cliente").rolling("24h", on="fecha_hora")["monto"].count().values - 1   # sin contar la actual
tx = tx.sort_values(["fecha_hora", "id_transaccion"]).reset_index(drop=True)
print(tx.groupby("es_fraude")["tx_24h"].mean().round(2).rename({0: "legítimas", 1: "fraude"}).to_string())

---
# Bloque C · Isolation Forest  ·  30 min

### La idea

Un **Isolation Forest** construye cientos de árboles con **cortes al azar**: elige una variable al azar y un punto de corte
al azar, y repite hasta que cada observación queda sola.

- Una compra **normal** está rodeada de compras parecidas: hacen falta **muchos cortes** para aislarla.
- Una compra **rara** está lejos de todas: queda sola en **pocos cortes**.

El score de anomalía es, básicamente, **qué tan rápido se aísla** la observación, promediado sobre todos los árboles. No usa
etiquetas: solo aprende cómo es lo normal.

### Demostración C.0 — Isolation Forest en dos dimensiones

In [ ]:
# ── DEMOSTRACIÓN C.0 ──────────────────────────────────────────────────
rng = np.random.default_rng(SEED)
X = np.r_[rng.normal(0, 1, (400, 2)), rng.normal([2.5, 2.5], 0.4, (80, 2)), [[4.5, -3], [-4, 3.5], [3.8, 4.8], [-3.5, -3.8]]]
iso = IsolationForest(n_estimators=200, random_state=SEED).fit(X)
xx, yy = np.meshgrid(np.linspace(-6, 6, 200), np.linspace(-6, 6, 200))
zz = -iso.score_samples(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
umbral = np.quantile(-iso.score_samples(X), 0.99)          # el 1 % más raro de los puntos

fig, ax = plt.subplots(figsize=(6.8, 5.8))
cs = ax.contourf(xx, yy, zz, levels=14, cmap="Blues", alpha=0.9)
ax.contour(xx, yy, zz, levels=[umbral], colors=MAG, linewidths=1.5, linestyles="--")
cb = plt.colorbar(cs, ax=ax, shrink=0.85); cb.set_label("score de anomalía (más alto = más raro)"); cb.outline.set_visible(False)
ax.scatter(X[:-4, 0], X[:-4, 1], s=12, color=NAVY, alpha=0.75, lw=0, label="compras normales")
ax.scatter(X[-4:, 0], X[-4:, 1], s=110, color=MAG, edgecolor="white", lw=1.5, zorder=3, label="anomalías")
ax.plot([], [], color=MAG, ls="--", label="frontera del 1 % más raro")
ax.set_aspect("equal"); ax.grid(False); ax.set_xlabel("variable 1"); ax.set_ylabel("variable 2")
ax.legend(loc="lower right", frameon=True, facecolor="white", framealpha=0.9, edgecolor=LINEA)
ax.set_title("Las zonas vacías se aíslan rápido")
subtitulo(ax, "Oscuro = pocos cortes al azar bastan para dejar el punto solo")
plt.tight_layout(); plt.show()

### Cómo se evalúa: precisión@k

El equipo de fraude **no revisa todas las alertas**: revisa las primeras **k** que entran en su capacidad. Lo que importa es
**cuántas de esas k son fraude**. Eso es la **precisión@k**, la misma lógica de las bandas de la Clase 9: ordenar por
score y mirar la parte de arriba.

Hoy usamos **k = 500** alertas para enero a septiembre de 2026 (unas dos por día).

### Ejercicio C.1 — Isolation Forest con variables crudas

1. `entreno`: compras de julio a diciembre de 2025. `prueba`: compras desde enero de 2026 (usa `.copy()`).
2. Entrena `IsolationForest(n_estimators=200, random_state=SEED, n_jobs=-1)` con `VARS_CRUDAS` en `entreno`.
3. Guarda en `prueba["score_crudas"]` el score de anomalía: **`-modelo.score_samples(...)`**. `score_samples` da valores
   más bajos para lo más raro; con el signo menos, más alto es más raro.

La función `precision_at_k` ya está escrita.

In [ ]:
def precision_at_k(y, score, k=500):
    orden = np.argsort(-np.asarray(score), kind="stable")[:k]
    return np.asarray(y)[orden].mean()

VARS_CRUDAS = ["log_monto", "madrugada", "extranjero", "online"]

# ── TU CÓDIGO ────────────────────────────────────────────────────────────
entreno = None
prueba = None
modelo_crudas = None
# prueba["score_crudas"] = ...

In [ ]:
# ── VERIFICACIÓN C.1 ──────────────────────────────────────────────────
ok_p = prueba is not None and "score_crudas" in prueba
r = [check("compras de entrenamiento", None if entreno is None else len(entreno), 193844, tol=0),
     check("compras de prueba", None if prueba is None else len(prueba), 245181, tol=0),
     check("precisión@500 con variables crudas", precision_at_k(prueba["es_fraude"], prueba["score_crudas"]) if ok_p else None, 0.552, tol=0.03)]
print()
print("C.1 OK" if all(r) else "Revisa C.1")

### Ejercicio C.2 — Isolation Forest con variables relativas

Repite con `VARS_REL` (las crudas más las cuatro relativas al cliente). Guarda el score en `prueba["score_rel"]` y calcula
`p_rel` (precisión@500) y `ap_rel` (`average_precision_score`, el área bajo la curva precisión-recall de la Clase 7).

In [ ]:
VARS_REL = VARS_CRUDAS + ["ratio_monto", "comercio_nuevo", "dispositivo_nuevo", "tx_24h"]

# ── TU CÓDIGO ────────────────────────────────────────────────────────────
modelo_rel = None
# prueba["score_rel"] = ...
p_rel = None
ap_rel = None

In [ ]:
# ── VERIFICACIÓN C.2 ──────────────────────────────────────────────────
r = [check("precisión@500 con variables relativas", p_rel, 0.808, tol=0.03),
     check("AP con variables relativas", ap_rel, 0.896, tol=0.02)]
print()
print("C.2 OK" if all(r) else "Revisa C.2")

**Mismo algoritmo, otras variables: de 55 % a 81 % de precisión.** Es la lección de la Clase 7 con otra cara: ahí el umbral
importaba más que el modelo; aquí, **las variables importan más que el algoritmo**.

### Demostración C.3 — Contra qué comparar: una regla de negocio y LOF

Un modelo de fraude siempre se compara con lo que el banco ya tiene: **reglas**. Probamos una razonable («monto mayor a 5
veces lo habitual y en el extranjero») y el **Local Outlier Factor (LOF)**, que mide qué tan aislado está un punto
respecto de sus vecinos más cercanos. LOF necesita variables escaladas y es más lento: lo entrenamos con una muestra
de 20 000 compras (tarda unos segundos).

In [ ]:
# ── DEMOSTRACIÓN C.3 ──────────────────────────────────────────────────
if prueba is not None and "score_rel" in prueba:
    t0 = time.time()
    esc = StandardScaler().fit(entreno[VARS_REL])
    muestra = entreno.sample(20000, random_state=SEED)
    lof = LocalOutlierFactor(n_neighbors=35, novelty=True).fit(esc.transform(muestra[VARS_REL]))
    prueba["score_lof"] = -lof.score_samples(esc.transform(prueba[VARS_REL]))
    print(f"LOF: {time.time() - t0:.0f} s")

    regla = (prueba["ratio_monto"] > 5) & (prueba["extranjero"] == 1)
    y = prueba["es_fraude"]; total = y.sum()
    def fila(nombre, alertas, aciertos, ap):
        return {"método": nombre, "alertas": alertas, "fraudes encontrados": aciertos,
                "precisión": aciertos / alertas, "recall": aciertos / total, "AP": ap}
    def top(col):
        return int(y[prueba[col].rank(ascending=False, method="first") <= 500].sum())
    comp = pd.DataFrame([
        fila("Regla: ratio > 5 y extranjero", int(regla.sum()), int(y[regla].sum()), np.nan),
        fila("Isolation Forest · crudas", 500, top("score_crudas"), average_precision_score(y, prueba["score_crudas"])),
        fila("Isolation Forest · relativas", 500, top("score_rel"), average_precision_score(y, prueba["score_rel"])),
        fila("LOF · relativas", 500, top("score_lof"), average_precision_score(y, prueba["score_lof"])),
    ]).set_index("método")
    print(comp.round(3).to_string())

    # curva precisión-recall: cada punto es «revisar las primeras k alertas»
    from sklearn.metrics import precision_recall_curve
    fig, ax = plt.subplots(figsize=(8, 4.8))
    for col, color, nombre in [("score_crudas", GRAY, "Isolation Forest · crudas"),
                               ("score_rel", BLUE, "Isolation Forest · relativas"),
                               ("score_lof", MAG, "LOF · relativas")]:
        p, rc, _ = precision_recall_curve(y, prueba[col])
        ax.plot(rc, p, color=color, lw=2, label=nombre)
        r500 = top(col) / total
        ax.plot(r500, top(col) / 500, "o", ms=8, color=color, mec="white", mew=1.5, zorder=4)
    ax.plot(comp.loc["Regla: ratio > 5 y extranjero", "recall"], comp.loc["Regla: ratio > 5 y extranjero", "precisión"],
            "D", ms=9, color=ORANGE, mec="white", mew=1.5, zorder=5, label="Regla: ratio > 5 y extranjero")
    ax.set_xlim(0, 1.01); ax.set_ylim(0, 1.03)
    ax.xaxis.set_major_formatter(PercentFormatter(1)); ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.set_xlabel("recall: % de los fraudes encontrados"); ax.set_ylabel("precisión: % de alertas que son fraude")
    ax.grid(True, axis="both")
    ax.legend(loc="lower left")
    ax.set_title("Precisión contra recall en enero–septiembre 2026")
    subtitulo(ax, "Los círculos marcan las 500 primeras alertas de cada modelo")
    plt.tight_layout(); plt.show()

Tres lecturas honestas:

1. **La regla no es mala.** Con 463 alertas encuentra 361 fraudes. Pero alguien tuvo que adivinar el 5 y la combinación,
   y solo agarra lo que se le ocurrió a quien la escribió.
2. **Isolation Forest con las mismas 500 alertas encuentra 404**, sin que nadie le diga qué buscar. Por eso los bancos usan
   **las dos**: reglas para lo conocido, anomalías para lo que nadie anticipó.
3. **LOF gana arriba (top 500) y pierde en el resto de la lista (AP)**, y tarda varias veces más. Isolation Forest es el
   punto de partida habitual por eso: rápido, sin escalar y estable.

### Demostración C.4 — Qué patrón se le escapa, y la trampa de `predict`

In [ ]:
# ── DEMOSTRACIÓN C.4 ──────────────────────────────────────────────────
if prueba is not None and "score_rel" in prueba:
    top500 = prueba.nlargest(500, "score_rel").merge(etiquetas[["id_transaccion", "patron"]], on="id_transaccion")
    recall_patron = (top500["patron"].value_counts() / etiquetas["patron"].value_counts()).round(3)
    print("Recall dentro de las 500 alertas, por patrón:"); print(recall_patron.to_string())
    alertas_predict = (modelo_rel.predict(prueba[VARS_REL]) == -1).sum()
    print(f"\nmodelo.predict() marca {alertas_predict:,} compras como anómalas (contamination='auto').".replace(",", " "))

    s = prueba["score_rel"]
    corte_500 = s.nlargest(500).min()
    corte_predict = -modelo_rel.offset_            # predict marca anomalía si el score supera este valor
    fig, ax = plt.subplots(figsize=(9, 4.2))
    bins = np.linspace(s.min(), s.max(), 70)
    ax.hist(s[prueba["es_fraude"] == 0], bins=bins, color=GRAY, alpha=0.6, label="legítimas")
    ax.hist(s[prueba["es_fraude"] == 1], bins=bins, color=MAG, alpha=0.85, label="fraude")
    ax.set_yscale("log"); ax.yaxis.set_major_formatter(MILES)
    for xv, color, txt in [(corte_predict, TINTA_2, f"predict(): {alertas_predict:,} alertas".replace(",", " ")),
                           (corte_500, NAVY, "top 500 alertas")]:
        ax.axvline(xv, color=color, lw=1.5, ls="--")
        ax.text(xv, ax.get_ylim()[1] * 0.5, f" {txt} " , color=color, fontsize=9, fontweight="bold",
                ha="left" if xv == corte_predict else "right")
    ax.set_xlabel("score de anomalía (Isolation Forest · relativas)"); ax.set_ylabel("compras (escala log)")
    ax.legend(loc="upper right")
    ax.set_title("Dónde cortar la lista de alertas")
    subtitulo(ax, "predict() corta mucho antes: decenas de miles de alertas que nadie revisa")
    plt.tight_layout(); plt.show()

- **Las tarjetas clonadas caen casi todas**: monto alto para el cliente, en el extranjero, de madrugada.
- **La prueba de tarjeta se escapa a la mitad.** Una compra de S/ 3 no tiene nada de raro *mirada sola*. Lo raro es
  **quién más** usó ese dispositivo. Eso no lo ve ninguna variable de la transacción: lo ve el **grafo** (Bloque D).
- **No uses `predict` tal cual.** Con `contamination="auto"` marca unas 30 mil compras: ningún equipo revisa eso.
  Se ordena por score y se corta por **capacidad** (k), igual que las bandas de la Clase 9.

---
# Bloque D · Grafos  ·  35 min

Un **grafo** tiene **nodos** (clientes, dispositivos, comercios) y **aristas** (quién usó qué). El fraude organizado deja
huella en la **estructura**: un mismo celular usado con decenas de tarjetas, o decenas de víctimas que pasaron por el mismo
comercio.

Usamos **NetworkX**, la librería estándar de grafos en Python.

### Ejercicio D.1 — El grafo cliente–dispositivo

Con las compras **online** hasta el 30 de junio de 2026:

1. `aristas`: los pares únicos `(id_cliente, id_dispositivo)`.
2. `G`: un `nx.Graph()` con los clientes y los dispositivos como nodos (atributo `tipo`) y una arista por par.
3. `grado_disp`: una `pd.Series` con el **grado** de cada dispositivo (cuántos clientes distintos lo usaron), ordenada de
   mayor a menor. Pista: `dict(G.degree(lista_de_dispositivos))`.

In [ ]:
CORTE_RED = pd.Timestamp("2026-07-01")

# ── TU CÓDIGO ────────────────────────────────────────────────────────────
aristas = None
G = None
grado_disp = None

In [ ]:
# ── VERIFICACIÓN D.1 ──────────────────────────────────────────────────
r = [check("nodos del grafo", None if G is None else G.number_of_nodes(), 13285, tol=0),
     check("aristas del grafo", None if G is None else G.number_of_edges(), 7575, tol=0),
     check("máximo de clientes en un dispositivo", None if grado_disp is None else grado_disp.max(), 32, tol=0),
     check("dispositivos con 3 o más clientes", None if grado_disp is None else (grado_disp >= 3).sum(), 6, tol=0)]
print()
print("D.1 OK" if all(r) else "Revisa D.1")

**7 317 dispositivos tienen un solo cliente y 68 tienen dos** (una pareja que comparte el celular o la laptop). Y hay **seis**
dispositivos usados por **11 a 32 clientes distintos**. Ninguna familia comparte un celular entre 32 personas.

### Demostración D.2 — Cómo se ve el anillo

In [ ]:
# ── DEMOSTRACIÓN D.2 ──────────────────────────────────────────────────
if grado_disp is not None:
    sosp = grado_disp[grado_disp >= 3].index.tolist()
    normales = grado_disp[grado_disp == 2].index[:12].tolist()
    nodos = set(sosp) | set(normales)
    for d in sosp + normales:
        nodos |= set(G.neighbors(d))
    H = G.subgraph(nodos)
    victimas = set(etiquetas["id_cliente"])
    pos = nx.spring_layout(H, seed=SEED, k=0.22, iterations=200)

    fig, ax = plt.subplots(figsize=(10, 7.5))
    nx.draw_networkx_edges(H, pos, edge_color=LINEA, width=0.8, ax=ax)
    clientes = [n for n in H if H.nodes[n]["tipo"] == "cliente"]
    grupos = [  # (nodos, color, tamaño, forma)
        ([n for n in clientes if n not in victimas], BLUE, 28, "o"),
        ([n for n in clientes if n in victimas], MAG, 28, "o"),
        (normales, GRAY, 90, "s"),
        (sosp, NAVY, 260, "s"),
    ]
    for lista, color, tam, forma in grupos:
        nx.draw_networkx_nodes(H, pos, nodelist=lista, node_color=color, node_size=tam, node_shape=forma,
                               edgecolors="white", linewidths=1, ax=ax)
    for d in sosp:
        x, y = pos[d]
        ax.annotate(f"{d}\n{grado_disp[d]} clientes", (x, y), xytext=(0, 14), textcoords="offset points",
                    ha="center", fontsize=8.5, fontweight="bold", color=NAVY,
                    bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="none", alpha=0.85))
    leyenda = [Line2D([], [], marker="s", ls="", ms=11, color=NAVY, label="dispositivo con 3+ clientes"),
               Line2D([], [], marker="s", ls="", ms=7, color=GRAY, label="dispositivo compartido en familia (2)"),
               Line2D([], [], marker="o", ls="", ms=6, color=MAG, label="cliente con fraude confirmado"),
               Line2D([], [], marker="o", ls="", ms=6, color=BLUE, label="cliente sin fraude")]
    ax.legend(handles=leyenda, loc="lower left", ncol=2, bbox_to_anchor=(0, -0.08))
    ax.set_title("El anillo: seis dispositivos usados con 11 a 32 tarjetas")
    subtitulo(ax, "Grafo cliente–dispositivo, compras online hasta junio 2026")
    ax.axis("off"); plt.tight_layout(); plt.show()

### Ejercicio D.3 — Los clientes conectados al anillo

1. `sospechosos`: la lista de dispositivos con `grado_disp >= 3`.
2. `clientes_red`: el **conjunto** de clientes conectados a algún dispositivo sospechoso (sus vecinos en `G`).
3. `en_fraude`: cuántos de `clientes_red` tienen algún fraude confirmado (`etiquetas["id_cliente"]`).

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
sospechosos = None
clientes_red = None
en_fraude = None

In [ ]:
# ── VERIFICACIÓN D.3 ──────────────────────────────────────────────────
r = [check("clientes conectados a dispositivos sospechosos", None if clientes_red is None else len(clientes_red), 122, tol=0),
     check("de ellos, con fraude confirmado", en_fraude, 122, tol=0)]
print()
print("D.3 OK" if all(r) else "Revisa D.3")

**Los 122 clientes conectados al anillo son víctimas.** Son, además, **todas** las víctimas de fraude online (comercio
comprometido y prueba de tarjeta). Una sola pregunta al grafo (¿qué dispositivo tiene demasiados clientes?) encuentra lo que
a Isolation Forest le costaba.

> **En la vida real no es tan limpio.** Hay dispositivos legítimos con muchos clientes: la computadora de una agencia de
> viajes, un cibercafé, una tablet en una tienda. El umbral (3 clientes) se calibra con el negocio, y el grafo da
> **candidatos para investigar**, no culpables.

### Ejercicio D.4 — Punto de compromiso: ¿dónde les robaron los datos?

Es **20 de marzo de 2026**. El banco tiene los reclamos que llegaron hasta ese día y quiere saber si hay un comercio donde
**se filtran las tarjetas**. La idea: si muchas víctimas pasaron por el mismo comercio **antes** de su primer cargo
desconocido, y eso no se explica por lo popular que es el comercio, ahí está la fuga.

1. `rec_corte`: los reclamos con `fecha_reclamo <= CORTE_REC`.
2. `visitas`: las compras **presenciales** de esos clientes en los **90 días anteriores** a su `fecha_primer_cargo`
   (une `tx` con `rec_corte` por `id_cliente` y filtra por fecha).
3. `cpp`: por comercio,
   - `victimas`: clientes distintos de `visitas`;
   - `pct_victimas`: `victimas / len(rec_corte)`;
   - `pct_base`: de **todos** los clientes que compraron en los 90 días antes del corte, qué fracción pasó por ese comercio
     (**ya está calculado**);
   - `esperadas = len(rec_corte) * pct_base`: cuántas víctimas habrían pasado por ahí **por casualidad**;
   - `exceso = victimas - esperadas`.

   Ordena por `exceso`, de mayor a menor.

In [ ]:
CORTE_REC = pd.Timestamp("2026-03-20")
ventana = tx[(tx["fecha_hora"] >= CORTE_REC - pd.Timedelta(days=90)) & (tx["fecha_hora"] < CORTE_REC)]
pct_base = ventana.groupby("id_comercio")["id_cliente"].nunique() / ventana["id_cliente"].nunique()

# ── TU CÓDIGO ────────────────────────────────────────────────────────────
rec_corte = None
visitas = None
cpp = None

In [ ]:
# ── VERIFICACIÓN D.4 ──────────────────────────────────────────────────
def celda_cpp(col):
    ok = cpp is not None and "C0225" in cpp.index and col in cpp
    return cpp.loc["C0225", col] if ok else None
r = [check("reclamos al 20/03/2026", None if rec_corte is None else len(rec_corte), 50, tol=0),
     check_bool("el comercio con más exceso es C0225", cpp is not None and len(cpp) > 0 and cpp.index[0] == "C0225",
                pista="¿ordenaste por exceso de mayor a menor?"),
     check("víctimas que pasaron por C0225", celda_cpp("victimas"), 15, tol=0),
     check("exceso de víctimas en C0225", celda_cpp("exceso"), 12.905, tol=0.01)]
print()
print("D.4 OK" if all(r) else "Revisa D.4")

**C0225, un grifo de Lima.** 15 de los 50 reclamantes pasaron por ahí (30 %), cuando lo esperable era el 4 %: unas
**13 víctimas de más**.

¿Por qué ordenar por `exceso` y no por `lift`? Mira la tabla de la demostración, ordenada por lift y con al menos 3
víctimas: un comercio de salud con **3 víctimas** tiene un lift de 30, porque casi nadie compra ahí. Con 3 casos eso puede ser casualidad. El exceso premia lo que es **raro y
grande a la vez**.

### Demostración D.5 — Cuánto vale encontrarlo el 20 de marzo

In [ ]:
# ── DEMOSTRACIÓN D.5 ──────────────────────────────────────────────────
if cpp is not None and {"victimas", "pct_victimas", "pct_base", "exceso"} <= set(cpp.columns):
    cpp["lift"] = cpp["pct_victimas"] / cpp["pct_base"]
    print(cpp[cpp["victimas"] >= 3].sort_values("lift", ascending=False).head(4)[["victimas", "pct_victimas", "pct_base", "lift", "exceso"]].round(3).to_string())
    visitantes = set(tx[(tx["id_comercio"] == "C0225") & (tx["fecha_hora"] < CORTE_REC)]["id_cliente"])
    a_reemitir = visitantes - set(rec_corte["id_cliente"])
    despues = tx[(tx["es_fraude"] == 1) & (tx["fecha_hora"] > CORTE_REC) & tx["id_cliente"].isin(a_reemitir)]
    print(f"\nTarjetas a bloquear y reemitir (pasaron por C0225 y aún no reclaman): {len(a_reemitir)}")
    print(f"Fraudes que se habrían evitado: {len(despues)} compras de {despues['id_cliente'].nunique()} clientes · S/ {despues['monto'].sum():,.0f}".replace(",", " "))

    # cada punto es un comercio: lo esperado por popularidad (x) contra lo observado en las víctimas (y)
    c = cpp.dropna(subset=["pct_base"])
    fig, ax = plt.subplots(figsize=(8.5, 5))
    xlim, ylim = c["pct_base"].max() * 1.15, c["pct_victimas"].max() * 1.15
    ax.plot([0, xlim], [0, xlim], color=TINTA_2, lw=1, ls="--")
    ax.text(xlim * 0.66, xlim * 0.66 - 0.006, "diagonal = lo esperado por casualidad", ha="left", va="top", fontsize=8.5, color=TINTA_2)
    otros = c.drop(index=["C0225"], errors="ignore")
    ax.scatter(otros["pct_base"], otros["pct_victimas"], s=otros["victimas"] * 18, color=GRAY, alpha=0.45,
               edgecolor="white", lw=0.8)
    if "C0225" in c.index:
        x0, y0 = c.loc["C0225", ["pct_base", "pct_victimas"]]
        ax.scatter(x0, y0, s=c.loc["C0225", "victimas"] * 18, color=MAG, edgecolor="white", lw=1.5, zorder=3)
        ax.vlines(x0, x0, y0, color=MAG, lw=1.5, ls=":")
        ax.annotate(f"C0225 · grifo en Lima\n{c.loc['C0225', 'victimas']:.0f} víctimas, {c.loc['C0225', 'esperadas']:.1f} esperadas",
                    (x0, y0), xytext=(14, -4), textcoords="offset points", fontsize=9, fontweight="bold", color=TINTA, va="center")
    top_lift = c[(c["victimas"] >= 3)].sort_values("lift", ascending=False).index[0]
    if top_lift != "C0225":
        x1, y1 = c.loc[top_lift, ["pct_base", "pct_victimas"]]
        ax.annotate(f"{top_lift} · lift {c.loc[top_lift, 'lift']:.0f}\npero solo {c.loc[top_lift, 'victimas']:.0f} víctimas",
                    (x1, y1), xytext=(40, 60), textcoords="offset points", fontsize=8.5, color=TINTA_2,
                    arrowprops=dict(arrowstyle="-", color=TINTA_2, lw=0.8))
    ax.set_xlim(0, xlim); ax.set_ylim(0, ylim)
    ax.xaxis.set_major_formatter(PercentFormatter(1, decimals=0)); ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
    ax.set_xlabel("% de todos los clientes que pasaron por el comercio (90 días)")
    ax.set_ylabel("% de los reclamantes que pasaron")
    ax.grid(True, axis="both")
    ax.set_title("Punto de compromiso al 20 de marzo de 2026")
    subtitulo(ax, "Tamaño = víctimas · la distancia vertical a la diagonal es el exceso")
    plt.tight_layout(); plt.show()

Bloquear y reemitir **252 tarjetas** el 20 de marzo habría evitado **138 compras fraudulentas por S/ 395 832** en 56
clientes. Las otras 196 tarjetas se reemiten sin que hubieran sufrido fraude: es el costo de la medida, y es muy chico
comparado con lo que se evita.

Esto es lo que se llama **análisis de punto de compromiso** y es de las tareas más rentables de un equipo de fraude: con
pocas víctimas confirmadas se protege a todas las que todavía no se enteran.

---
# Bloque E · El grafo dentro del modelo  ·  10 min

El grafo no compite con Isolation Forest: **le da variables**. La más simple es `clientes_disp`: cuántos clientes distintos
había usado el dispositivo **hasta esa compra** (sin mirar el futuro, como siempre).

### Ejercicio E.1 — Isolation Forest con una variable de grafo

La variable ya está calculada en la celda. Agrega `"clientes_disp"` a `VARS_REL`, entrena otro Isolation Forest y calcula
`p_grafo` (precisión@500) y `ap_grafo`.

In [ ]:
# variable de grafo en el tiempo: k-ésimo cliente distinto del dispositivo
d = tx[tx["id_dispositivo"].notna()][["id_transaccion", "id_cliente", "id_dispositivo", "fecha_hora"]]
primeras = d.drop_duplicates(["id_dispositivo", "id_cliente"])[["id_dispositivo", "fecha_hora"]].copy()
primeras["clientes_disp"] = primeras.groupby("id_dispositivo").cumcount() + 1
d = pd.merge_asof(d.sort_values("fecha_hora"), primeras.sort_values("fecha_hora"), on="fecha_hora", by="id_dispositivo")
tx["clientes_disp"] = tx["id_transaccion"].map(d.set_index("id_transaccion")["clientes_disp"]).fillna(0)
if prueba is not None:
    prueba["clientes_disp"] = tx.set_index("id_transaccion").loc[prueba["id_transaccion"], "clientes_disp"].values
    entreno = tx[tx["fecha_hora"] < INICIO_2026]
VARS_GRAFO = VARS_REL + ["clientes_disp"]

# ── TU CÓDIGO ────────────────────────────────────────────────────────────
modelo_grafo = None
# prueba["score_grafo"] = ...
p_grafo = None
ap_grafo = None

In [ ]:
# ── VERIFICACIÓN E.1 ──────────────────────────────────────────────────
r = [check("precisión@500 con la variable de grafo", p_grafo, 0.884, tol=0.03),
     check("AP con la variable de grafo", ap_grafo, 0.943, tol=0.02)]
print()
print("E.1 OK" if all(r) else "Revisa E.1")

In [ ]:
# ── DEMOSTRACIÓN E.2 ──────────────────────────────────────────────────
MODELOS = [("score_crudas", "crudas", GRAY), ("score_rel", "+ relativas", BLUE), ("score_grafo", "+ grafo", MAG)]
if prueba is not None and all(col in prueba for col, _, _ in MODELOS):
    y = prueba["es_fraude"]
    res = pd.DataFrame({nombre: [precision_at_k(y, prueba[col], k) for k in (250, 500, 1000)] for col, nombre, _ in MODELOS},
                       index=["k = 250", "k = 500", "k = 1000"])
    print(res.round(3).to_string())
    rec = {}
    for col, nombre, _ in MODELOS:
        t5 = prueba.nlargest(500, col).merge(etiquetas[["id_transaccion", "patron"]], on="id_transaccion")
        rec[nombre] = t5["patron"].value_counts() / etiquetas["patron"].value_counts()
    rec = pd.DataFrame(rec).fillna(0).reindex(list(COLOR_PATRON))
    print("\nRecall por patrón en las 500 primeras alertas:")
    print(rec.round(3).to_string())

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={"width_ratios": [1.15, 1]})
    # izquierda: precisión según cuántas alertas revisa el equipo
    ks = np.arange(25, 1501, 25)
    for col, nombre, color in MODELOS:
        pk = [precision_at_k(y, prueba[col], k) for k in ks]
        ax1.plot(ks, pk, color=color, lw=2, label=nombre)
        ax1.plot(500, precision_at_k(y, prueba[col], 500), "o", ms=8, color=color, mec="white", mew=1.5, zorder=4)
    ax1.plot(ks, np.minimum(1, y.sum() / ks), color=TINTA_2, lw=1, ls=":")
    ax1.text(1150, y.sum() / 1150 + 0.05, "techo: solo hay\n%d fraudes" % y.sum(), fontsize=8.5, color=TINTA_2)
    ax1.axvline(500, color=LINEA, lw=1)
    ax1.legend(loc="lower left")
    ax1.set_xlim(0, 1520); ax1.set_ylim(0, 1.05); ax1.yaxis.set_major_formatter(PercentFormatter(1))
    ax1.set_xlabel("k: alertas que revisa el equipo"); ax1.set_ylabel("precisión@k")
    ax1.set_title("Precisión según la capacidad del equipo")
    subtitulo(ax1, "Círculos: k = 500")
    # derecha: recall por patrón
    w = 0.26; xb = np.arange(len(rec))
    for j, (_, nombre, color) in enumerate(MODELOS):
        barras = ax2.bar(xb + (j - 1) * w, rec[nombre], width=w - 0.03, color=color, label=nombre)
        if nombre != "crudas":
            ax2.bar_label(barras, labels=[f"{v:.0%}" for v in rec[nombre]], fontsize=8, padding=2, color=TINTA)
    ax2.set_xticks(xb, [NOMBRE_PATRON[p].replace(" ", "\n", 1) for p in rec.index]); ax2.tick_params(axis="x", length=0)
    ax2.set_ylim(0, 1.25); ax2.yaxis.set_major_formatter(PercentFormatter(1))
    ax2.set_ylabel("recall en las 500 alertas"); ax2.legend(loc="upper left", ncol=3)
    ax2.set_title("Qué patrón encuentra cada modelo")
    subtitulo(ax2, "La variable de grafo rescata la prueba de tarjeta")
    plt.tight_layout(); plt.show()

De **55 % a 81 % a 88 %** con el mismo algoritmo, y la prueba de tarjeta, que se escapaba a la mitad, pasa de 55 % a
77 % de recall. Cada salto vino de una pregunta distinta: ¿es raro **para este cliente**?
y ¿con **quién** está conectado?

---
# Cierre

### Los números del día

| | |
|---|---|
| Tasa de fraude | 0.12 % (516 de 439 025 compras) · 1 de cada 851 |
| Isolation Forest, precisión@500 | 55 % crudas · 81 % relativas al cliente · 88 % con variable de grafo |
| Regla «ratio > 5 y extranjero» | 463 alertas, 78 % de precisión |
| Anillo de dispositivos | 6 dispositivos con 11 a 32 clientes · 122 clientes conectados, todos víctimas |
| Punto de compromiso al 20/03 | grifo C0225: 15 de 50 reclamantes (30 % contra 4 % esperado) |
| Valor de encontrarlo | 252 tarjetas reemitidas evitan S/ 395 832 en 138 compras fraudulentas |

### Checklist de salida

- [ ] Sé por qué el fraude se aborda como **anomalía**: pocas etiquetas, tardías y un patrón que cambia.
- [ ] Construyo variables **relativas al cliente** usando solo su pasado.
- [ ] Entreno un Isolation Forest y lo evalúo con **precisión@k**, no con `predict`.
- [ ] Comparo el modelo contra una **regla de negocio** y contra LOF.
- [ ] Armo un grafo con NetworkX y encuentro dispositivos compartidos por el **grado**.
- [ ] Hago un análisis de **punto de compromiso** con exceso de víctimas sobre lo esperado.
- [ ] Llevo una variable de grafo al modelo, **calculada en el tiempo**.

### Preguntas tipo examen

**1.** ¿Por qué Isolation Forest no necesita etiquetas para entrenar, y para qué se usan entonces las etiquetas?

> **Respuesta.** Porque aprende cómo es lo normal y puntúa qué tan fácil se aísla cada compra con cortes al azar. Las
> etiquetas (los fraudes confirmados) solo sirven para evaluar: medir cuántas de las primeras k alertas son fraude.

**2.** Un analista entrena Isolation Forest y usa `predict()`: le salen 30 mil alertas en nueve meses. ¿Qué hizo mal?

> **Respuesta.** `predict` corta con un umbral por defecto (`contamination="auto"`) que no conoce la capacidad del equipo.
> Hay que ordenar por score y quedarse con las k alertas que el equipo puede revisar.

**3.** Una compra de S/ 3 en una tienda online no tiene nada de raro. ¿Cómo la detectas si es una prueba de tarjeta?

> **Respuesta.** Por la red: el dispositivo desde el que se hizo ya se usó con muchas otras tarjetas. Esa información no
> está en la transacción; sale del grafo cliente–dispositivo.

**4.** En el análisis de punto de compromiso, un comercio tiene lift 30 con 3 víctimas y otro lift 7 con 15. ¿Cuál
investigas primero?

> **Respuesta.** El de 15. Con 3 casos el lift alto puede ser casualidad (el comercio casi no tiene clientes). El exceso
> de víctimas sobre lo esperado (13 contra 3) muestra cuál concentra de verdad a las víctimas.

**5.** ¿Por qué `clientes_disp` se calcula «hasta esa compra» y no con todo el periodo?

> **Respuesta.** Para no mirar el futuro. Con todo el periodo, la primera compra de un anillo ya «sabría» que el
> dispositivo terminará con 32 clientes, algo que el banco no sabe en ese momento. Es la misma fuga de información de la
> Clase 12.

### Lo que sigue

Próxima clase: **puesta en producción e introducción a Databricks**.